In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss, roc_auc_score

In [ ]:
train_set=pd.read_csv("/content/Train.csv")
test_set=pd.read_csv("/content/Test.csv")
prior_set=pd.read_csv("/content/Prior.csv")
data_dict=pd.read_csv("/content/dataset_data_dictionary.csv")

In [ ]:
test_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5621 entries, 0 to 5620
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ID                     5621 non-null   object
 1   farmer_name            5621 non-null   object
 2   training_day           5621 non-null   object
 3   gender                 5621 non-null   object
 4   registration           5621 non-null   object
 5   age                    5621 non-null   object
 6   group_name             5621 non-null   object
 7   belong_to_cooperative  5621 non-null   int64 
 8   county                 5621 non-null   object
 9   subcounty              5621 non-null   object
 10  ward                   5621 non-null   object
 11  has_topic_trained_on   5621 non-null   int64 
 12  trainer                5621 non-null   object
 13  topics_list            5621 non-null   object
dtypes: int64(2), object(12)
memory usage: 614.9+ KB


In [ ]:
test_set["training_day"].max(), test_set["training_day"].min()

('2025-12-12', '2025-05-02')

In [ ]:
train_set["training_day"].max(), train_set["training_day"].min()

('2025-04-12', '2024-01-03')

In [ ]:
prior_set["training_day"].max(), prior_set["training_day"].min()

('2025-12-11', '2024-01-03')

In [ ]:
train_set.groupby("registration")["adopted_within_07_days"].mean()

,adopted_within_07_days
registration,
Manual,0.006598
Ussd,0.014195


# **FEATURE 1: IS_USSD**

### **DATE CONVERSION**

In [ ]:
prior_set["training_day"] = pd.to_datetime(prior_set["training_day"])
train_set["training_day"]  = pd.to_datetime(train_set["training_day"])
test_set["training_day"]   = pd.to_datetime(test_set["training_day"])

In [ ]:
history_set = pd.concat([prior_set, train_set], ignore_index=True)
history_set

,ID,farmer_name,training_day,gender,registration,age,group_name,belong_to_cooperative,county,subcounty,ward,adopted_within_07_days,adopted_within_90_days,adopted_within_120_days,has_topic_trained_on,trainer,topics_list
0,ID_70GP6F,FAR_leopgvh,2024-01-03,Female,Manual,Above 35,GRP_yvpakgc,0,CNT_lpotuu,SUB_lpotuuf,WRD_lpotuufh,0,0,0,0,TRA_szrwyfzz,"['Ndume App', 'Poultry Feeding']"
1,ID_IWQOWJ,FAR_vdcjfxm,2024-01-03,Female,Ussd,Above 35,GRP_yvpakgc,0,CNT_lpotuu,SUB_lpotuuf,WRD_lpotuufh,0,0,0,0,TRA_szrwyfzz,"['Ndume App', 'Poultry Feeding']"
2,ID_Z3ES85,FAR_hfkybdg,2024-01-03,Female,Manual,Above 35,GRP_zmblxsw,0,CNT_fhdsoy,SUB_mdyljqn,WRD_atkhhvon,0,0,0,1,TRA_rkvyofbh,"['Asili Fertilizer (Organic)', 'Biosecurity In..."
3,ID_JNZM6R,FAR_hfkybdg,2024-01-03,Female,Manual,Above 35,GRP_psdrfni,0,CNT_fhdsoy,SUB_mdyljqn,WRD_atkhhvon,0,0,0,1,TRA_rkvyofbh,['Poultry Products']
4,ID_BNJ1GU,FAR_hfkybdg,2024-01-03,Female,Manual,Above 35,GRP_psdrfni,1,CNT_fhdsoy,SUB_mdyljqn,WRD_atkhhvon,0,0,0,1,TRA_rkvyofbh,['Record Keeping In Dairy']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58413,ID_1WULAD,FAR_ixfwumd,2025-04-12,Male,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']]
58414,ID_PSPMAI,FAR_djalgnz,2025-04-12,Male,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']]
58415,ID_VD58N6,FAR_plzqhhs,2025-04-12,Female,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']]
58416,ID_TPZS0X,FAR_tvstqsy,2025-04-12,Female,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']]


In [ ]:
prior_farmers=set(prior_set["farmer_name"].unique())
train_farmers=set(train_set["farmer_name"].unique())
overlap=train_farmers.intersection(prior_farmers)
len(overlap), len(train_farmers), len(prior_farmers)

(3193, 13536, 6719)

## **COUNTY EDA**

In [ ]:
prior_set.groupby("county")["adopted_within_07_days"].mean()

,adopted_within_07_days
county,
CNT_fhdsoy,0.038992
CNT_fnywiv,0.000000
CNT_hfhimd,0.000000
CNT_lpotuu,0.004831
CNT_mggvna,0.008197
CNT_mvqqmw,0.028388
CNT_rihpna,0.000000
CNT_vvuukv,0.010633
CNT_yljqnl,0.000792


In [ ]:
county_counts = prior_set.groupby('county').size()
county_adoption = prior_set.groupby('county')['adopted_within_07_days'].mean()
county_analysis = pd.DataFrame({'count': county_counts,'adoption_rate': county_adoption}).sort_values('adoption_rate', ascending=False)
county_analysis

,count,adoption_rate
county,,
CNT_fhdsoy,7540,0.038992
CNT_mvqqmw,5284,0.028388
CNT_vvuukv,19656,0.010633
CNT_mggvna,366,0.008197
CNT_lpotuu,207,0.004831
CNT_yljqnl,8834,0.000792
CNT_hfhimd,130,0.000000
CNT_fnywiv,1,0.000000
CNT_rihpna,2864,0.000000


## **FEATURE ENGINEERING 2: COUNTY ADOPTION**

In [ ]:
train_set["county_adoption_07"]=train_set["county"].map(prior_set.groupby('county')['adopted_within_07_days'].mean())
train_set["county_adoption_90"]=train_set["county"].map(prior_set.groupby('county')['adopted_within_90_days'].mean())
train_set["county_adoption_120"]=train_set["county"].map(prior_set.groupby('county')['adopted_within_120_days'].mean())

In [ ]:
train_set["county_adoption_07"].min(), train_set["county_adoption_07"].max()

(0.0, 0.0389920424403183)

In [ ]:
prior_set.groupby(["trainer", "county"])["adopted_within_07_days"].agg(["mean", "size"])

mean   size
trainer      county                     
TRA_dttdgplk CNT_fhdsoy  0.000000      4
TRA_gertumxc CNT_vvuukv  0.010633  19656
TRA_hyodnntj CNT_mggvna  0.008197    366
TRA_kkzpfdtu CNT_mvqqmw  0.028388   5284
TRA_rkvyofbh CNT_fhdsoy  0.039013   7536
             CNT_fnywiv  0.000000      1
TRA_suiifsur CNT_yljqnl  0.000792   8834
TRA_szrwyfzz CNT_lpotuu  0.004831    207
TRA_twwcfcum CNT_hfhimd  0.000000      3
TRA_ubcgvofe CNT_hfhimd  0.000000    127
             CNT_rihpna  0.000000   2864

In [ ]:
train_set

,ID,farmer_name,training_day,gender,registration,age,group_name,belong_to_cooperative,county,subcounty,ward,adopted_within_07_days,adopted_within_90_days,adopted_within_120_days,has_topic_trained_on,trainer,topics_list,county_adoption_07,county_adoption_90,county_adoption_120
0,ID_CENCC8,FAR_eqbhscj,2024-01-03,Female,Manual,Above 35,GRP_yvpakgc,0,CNT_lpotuu,SUB_lpotuuf,WRD_lpotuufh,0,0,0,0,['TRA_szrwyfzz'],"[['Ndume App', 'Poultry Feeding']]",0.004831,0.009662,0.024155
1,ID_YTO0FF,FAR_qlwtyik,2024-01-03,Female,Manual,Above 35,GRP_zemrbsy,1,CNT_fhdsoy,SUB_mdyljqn,WRD_atkhhvon,0,0,0,1,['TRA_rkvyofbh'],"[['Poultry Housing'], ['Poultry Housing']]",0.038992,0.075199,0.098011
2,ID_1476PE,FAR_somfzxp,2024-01-03,Female,Manual,Above 35,GRP_zmblxsw,0,CNT_fhdsoy,SUB_mdyljqn,WRD_atkhhvon,0,0,0,1,['TRA_rkvyofbh'],"[['Asili Fertilizer (Organic)', 'Biosecurity I...",0.038992,0.075199,0.098011
3,ID_MLKLIR,FAR_ongcqyd,2024-01-03,Female,Manual,Above 35,GRP_psdrfni,0,CNT_fhdsoy,SUB_mdyljqn,WRD_atkhhvon,0,0,0,1,['TRA_rkvyofbh'],"[['Poultry Products'], ['Record Keeping In Dai...",0.038992,0.075199,0.098011
4,ID_V5ZVTA,FAR_ztsbhhm,2024-01-03,Female,Ussd,Below 35,GRP_yvpakgc,0,CNT_lpotuu,SUB_lpotuuf,WRD_lpotuufh,0,0,0,0,['TRA_szrwyfzz'],"[['Ndume App', 'Poultry Feeding']]",0.004831,0.009662,0.024155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13531,ID_1WULAD,FAR_ixfwumd,2025-04-12,Male,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']],0.000000,0.004190,0.006983
13532,ID_PSPMAI,FAR_djalgnz,2025-04-12,Male,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']],0.000000,0.004190,0.006983
13533,ID_VD58N6,FAR_plzqhhs,2025-04-12,Female,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']],0.000000,0.004190,0.006983
13534,ID_TPZS0X,FAR_tvstqsy,2025-04-12,Female,Manual,Above 35,GRP_lwsitwv,0,CNT_rihpna,SUB_twzhvtx,WRD_wdjchnle,0,0,0,1,['TRA_ubcgvofe'],[['Digital Finance With Kcb']],0.000000,0.004190,0.006983


## **FEATURE ENGINEERING 3: STRIP OFF TRAINER**

In [ ]:
train_set["trainer"] = train_set["trainer"].str.strip("[]'")
train_set["trainer"]

,trainer
0,TRA_szrwyfzz
1,TRA_rkvyofbh
2,TRA_rkvyofbh
3,TRA_rkvyofbh
4,TRA_szrwyfzz
...,...
13531,TRA_ubcgvofe
13532,TRA_ubcgvofe
13533,TRA_ubcgvofe
13534,TRA_ubcgvofe


In [ ]:
for col in ["trainer", "county", "group_name"]:
    prior_vals = set(prior_set[col].unique())
    train_vals = set(train_set[col].unique())
    overlap_ = train_vals.intersection(prior_vals)
    print(f"{col}: {len(overlap_)}/{len(train_vals)} = {len(overlap_)/len(train_vals):.0%}")

trainer: 9/12 = 75%
county: 8/8 = 100%
group_name: 706/864 = 82%


In [ ]:
for col in ["trainer", "county", "group_name"]:
    test_vals = set(test_set[col].unique())
    train_vals = set(train_set[col].unique())
    overlap__ = test_vals.intersection(train_vals)
    print(f"{col}: {len(overlap__)}/{len(test_vals)} = {len(overlap__)/len(test_vals):.0%}")

trainer: 0/7 = 0%
county: 7/7 = 100%
group_name: 126/239 = 53%


In [ ]:
prior_set["trainer"].value_counts()

,count
trainer,
TRA_gertumxc,19656
TRA_suiifsur,8834
TRA_rkvyofbh,7537
TRA_kkzpfdtu,5284
TRA_ubcgvofe,2991
TRA_hyodnntj,366
TRA_szrwyfzz,207
TRA_dttdgplk,4
TRA_twwcfcum,3


In [ ]:
train_set["trainer"].value_counts()

,count
trainer,
TRA_gertumxc,3049
TRA_suiifsur,2190
TRA_hyodnntj,2168
TRA_szrwyfzz,2027
TRA_rkvyofbh,1877
TRA_ubcgvofe,1456
TRA_kkzpfdtu,732
TRA_twwcfcum,24
TRA_dttdgplk,10


In [ ]:
prior_set.groupby("trainer")["adopted_within_07_days"].agg(["mean","size"])

,mean,size
trainer,,
TRA_dttdgplk,0.000000,4
TRA_gertumxc,0.010633,19656
TRA_hyodnntj,0.008197,366
TRA_kkzpfdtu,0.028388,5284
TRA_rkvyofbh,0.039008,7537
TRA_suiifsur,0.000792,8834
TRA_szrwyfzz,0.004831,207
TRA_twwcfcum,0.000000,3
TRA_ubcgvofe,0.000000,2991


## **FEATURE ENGINEERING 4:SMOOTHED ENCODING ON TRAINER: BAYESIAN SMOOTHING**
smoothed_rate = (count × group_mean + m × global_mean) / (count + m)

In [ ]:
def smoothed_target_encoding(prior_df, train_df, group_col, target_col, m=20):
    global_mean = prior_df[target_col].mean()
    agg = prior_df.groupby(group_col)[target_col].agg(["mean", "count"])
    agg["smoothed"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    return train_df[group_col].map(agg["smoothed"])

for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    train_set[f"trainer_adoption_{short}"] = smoothed_target_encoding(prior_set, train_set, "trainer", target, m=20)

In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    global_mean = prior_set[target].mean()
    train_set[f"trainer_adoption_{short}"] = train_set[f"trainer_adoption_{short}"].fillna(global_mean)

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **TRAINER+TOPIC**

## **GROUP NAME**

In [ ]:
prior_set.groupby("group_name")["adopted_within_07_days"].agg(["mean", "size"]).sort_values("mean", ascending=False).head(20)

,mean,size
group_name,,
GRP_wftrnca,1.000000,3
GRP_krnvndq,0.562500,64
GRP_gvbkzci,0.500000,2
GRP_lwobxlm,0.375000,32
GRP_siopuwg,0.352941,51
GRP_dzwjehf,0.338462,65
GRP_xruwxnp,0.266667,15
GRP_xxglbfb,0.250000,12
GRP_fmpjwna,0.250000,12


In [ ]:
prior_set.groupby("group_name")["adopted_within_07_days"].agg(["mean", "size"]).sort_values("size", ascending=False).head(20)

,mean,size
group_name,,
GRP_junsdbz,0.028897,1142
GRP_dkssfll,0.016651,1081
GRP_quagidu,0.003175,945
GRP_wdumlmm,0.000000,903
GRP_scvwssr,0.043084,882
GRP_pzaumib,0.000000,859
GRP_agfainw,0.023136,778
GRP_diinohf,0.045093,754
GRP_eppchjm,0.000000,744


## **FEATURE ENGINEERING 4 ON GROUP_NAME: BAYESIAN SMOOTHING**
 smoothed_rate = (count × group_mean + m × global_mean) / (count + m)

In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    train_set[f"group_adoption_{short}"] = smoothed_target_encoding(prior_set, train_set, "group_name", target, m=20)

In [ ]:
train_set[["group_name", "group_adoption_07", "group_adoption_90", "group_adoption_120"]].drop_duplicates().sort_values("group_adoption_07", ascending=False).head(10)

,group_name,group_adoption_07,group_adoption_90,group_adoption_120
7071,GRP_krnvndq,0.432094,0.436677,0.439643
4965,GRP_dzwjehf,0.262305,0.266834,0.281529
11200,GRP_ziubiwx,0.171864,0.311021,0.313820
10073,GRP_modcvbx,0.164605,0.592736,0.595252
7594,GRP_hmlgsfp,0.163940,0.174124,0.367132
11109,GRP_sncbegx,0.133651,0.137836,0.140543
9215,GRP_xqxgwyy,0.132808,0.144853,0.221869
12852,GRP_xruwxnp,0.122740,0.133740,0.140857
8012,GRP_txssjmu,0.113050,0.123181,0.129737
10146,GRP_gnhksrt,0.103999,0.138191,0.181111


In [ ]:
prior_set.groupby(["group_name", "gender"])["adopted_within_07_days"].agg(["mean", "size"]).sort_values("mean", ascending=False).head(20)

,,mean,size
group_name,gender,,
GRP_gvbkzci,Male,1.000000,1
GRP_hmlgsfp,Male,1.000000,2
GRP_wftrnca,Female,1.000000,3
GRP_eldhpyw,Male,0.750000,4
GRP_krnvndq,Female,0.562500,64
GRP_umcqiju,Female,0.500000,2
GRP_lwobxlm,Female,0.500000,24
GRP_siopuwg,Female,0.352941,51
GRP_dzwjehf,Female,0.338462,65


Gender +group name is not adding anything

In [ ]:
# prior_set["topics_list"].str.strip("[]")
prior_set.groupby(["group_name", "topics_list"])["adopted_within_07_days"].agg(["mean", "size"]).sort_values("mean", ascending=False).head(20)

mean  size
group_name  topics_list                                                       
GRP_lrypvns ['Poultry Feeding With Tyari']                      1.000000     1
GRP_yiavgqw ['Biosecurity In Poultry Farming', 'Calf Feedin...  1.000000     1
GRP_ofeyftj ['Benfits Of Sistema Biogas', 'Dairy Nutrition ...  1.000000     1
GRP_lrypvns ['Dairy Nutrition With Tyari']                      1.000000     1
GRP_dzwjehf ['Dairy Nutrition With Tyari', 'Importance Of V...  1.000000     5
            ['Dairy Nutrition With Tyari', 'Importance Of V...  1.000000     2
GRP_zmljhtp ['Poultry Health With Biodeal']                     1.000000     1
            ['Importance Of Mineral Supplementation']           1.000000     1
            ['Herd. Health. Management']                        1.000000     1
GRP_axedqjj ['Asili Fertilizer (Organic)', 'Benefits Of Sis...  1.000000     1
GRP_wftrnca ['Dairy Nutrition With Tyari', 'Importance Of V...  1.000000     1
            ['Dairy Nutrition With Tyari']                      1.000000     1
GRP_axedqjj ['Benefits Of Sistema Biogas', 'Benfits Of Sist...  1.000000     1
GRP_iegtpjx ['Benfits Of Sistema Biogas', 'Control Of Exter...  1.000000     1
GRP_wftrnca ['Poultry Feeding With Tyari']                      1.000000     1
GRP_axedqjj ['Asili Fertilizer (Organic)', 'Benfits Of Sist...  1.000000     1
GRP_hmlgsfp ['Poultry Feeding With Tyari']                      0.833333     6
            ['Dairy Nutrition With Tyari']                      0.833333     6
GRP_dzwjehf ['Poultry Feeding With Tyari']                      0.800000    10
            ['Dairy Nutrition With Tyari']                      0.777778     9

### **RETURN TO GROUP +TOPIC LATER**

## **FARMER_NAME**

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

In [ ]:
farmer_counts = prior_set.groupby("farmer_name").size().reset_index(name="farmer_training_count")
train_set = train_set.merge(farmer_counts, on="farmer_name", how="left")
train_set["farmer_training_count"] = train_set["farmer_training_count"].fillna(0)

It means how many times this farmer has attended any training before the current one. The intuition is engagement level. A farmer who has shown up 20 times before is more committed to learning than one showing up for the first time. Commitment correlates with adoption. That's the feature.

In [ ]:
farmer_ever = prior_set.groupby("farmer_name")["adopted_within_07_days"].sum().reset_index()
farmer_ever["farmer_ever_adopted_07"] = (farmer_ever["adopted_within_07_days"] > 0).astype(int)
train_set = train_set.merge(farmer_ever[["farmer_name", "farmer_ever_adopted_07"]], on="farmer_name", how="left")
train_set["farmer_ever_adopted_07"].fillna(0, inplace=True)

/tmp/ipykernel_399/2156007604.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_set["farmer_ever_adopted_07"].fillna(0, inplace=True)


In [ ]:
farmer_ever_90 = prior_set.groupby("farmer_name")["adopted_within_90_days"].sum().reset_index()
farmer_ever_90["farmer_ever_adopted_90"] = (farmer_ever_90["adopted_within_90_days"] > 0).astype(int)
train_set = train_set.merge(farmer_ever_90[["farmer_name", "farmer_ever_adopted_90"]], on="farmer_name", how="left")
train_set["farmer_ever_adopted_90"].fillna(0)

farmer_ever_120 = prior_set.groupby("farmer_name")["adopted_within_120_days"].sum().reset_index()
farmer_ever_120["farmer_ever_adopted_120"] = (farmer_ever_120["adopted_within_120_days"] > 0).astype(int)
train_set = train_set.merge(farmer_ever_120[["farmer_name", "farmer_ever_adopted_120"]], on="farmer_name", how="left")
train_set["farmer_ever_adopted_120"].fillna(0)

,farmer_ever_adopted_120
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
13531,0.0
13532,0.0
13533,0.0
13534,0.0


In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **FILLING THE NULL VALUES**

In [ ]:
global_mean_07 = prior_set["adopted_within_07_days"].mean()
train_set["group_adoption_07"] = train_set["group_adoption_07"].fillna(global_mean_07)

global_mean_90 = prior_set["adopted_within_90_days"].mean()
train_set["group_adoption_90"] = train_set["group_adoption_90"].fillna(global_mean_90)

global_mean_120 = prior_set["adopted_within_120_days"].mean()
train_set["group_adoption_120"] = train_set["group_adoption_120"].fillna(global_mean_120)

train_set["farmer_ever_adopted_90"] = train_set["farmer_ever_adopted_90"].fillna(0)
train_set["farmer_ever_adopted_120"] = train_set["farmer_ever_adopted_120"].fillna(0)


In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **FARMER ROLLING RATE**

In [ ]:
farmer_rate = prior_set.groupby("farmer_name")["adopted_within_07_days"].agg(["sum", "count", "mean"])
farmer_rate.columns = ["total_adoptions", "total_trainings", "rolling_rate"]
farmer_rate = farmer_rate[farmer_rate["total_trainings"] >= 2]
farmer_rate.sort_values("rolling_rate", ascending=False).head(20)

,total_adoptions,total_trainings,rolling_rate
farmer_name,,,
FAR_qdwatnt,3,3,1.0
FAR_kjkkmoa,2,2,1.0
FAR_elkcmzh,3,3,1.0
FAR_qfivsxs,2,2,1.0
FAR_kpddsil,3,3,1.0
FAR_grntzog,2,2,1.0
FAR_tchpjjc,3,3,1.0
FAR_rkltbtz,2,2,1.0
FAR_tsngidr,2,2,1.0


In [ ]:
temp_rolling = pd.DataFrame()
temp_rolling["farmer_rolling_rate_07"] = train_set["farmer_name"].map(
    prior_set.groupby("farmer_name")["adopted_within_07_days"].mean()
)
temp_rolling["adopted_07"] = train_set["adopted_within_07_days"]

temp_rolling.groupby(pd.cut(temp_rolling["farmer_rolling_rate_07"], bins=5))["adopted_07"].agg(["mean", "size"])

/tmp/ipykernel_399/2634143398.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp_rolling.groupby(pd.cut(temp_rolling["farmer_rolling_rate_07"], bins=5))["adopted_07"].agg(["mean", "size"])


,mean,size
farmer_rolling_rate_07,,
"(-0.001, 0.2]",0.014949,3144
"(0.2, 0.4]",0.074074,27
"(0.4, 0.6]",0.200000,5
"(0.6, 0.8]",0.000000,3
"(0.8, 1.0]",0.285714,14


In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    train_set[f"farmer_rolling_rate_{short}"] = smoothed_target_encoding(prior_set, train_set, "farmer_name", target, m=20)

train_set[["farmer_name", "farmer_rolling_rate_07"]].drop_duplicates().sort_values("farmer_rolling_rate_07", ascending=False).head(10)

,farmer_name,farmer_rolling_rate_07
8407,FAR_jijgafp,0.171835
3278,FAR_qeelqxm,0.159107
12930,FAR_lmyfetp,0.159107
4468,FAR_cdmnaka,0.143299
2643,FAR_zdtqtiy,0.143299
2665,FAR_elkcmzh,0.143299
10222,FAR_kaqbhzu,0.143299
10276,FAR_tchpjjc,0.143299
12538,FAR_qdwatnt,0.143299
4020,FAR_sqgphnl,0.143299


In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    global_mean = prior_set[target].mean()
    train_set[f"farmer_rolling_rate_{short}"] = train_set[f"farmer_rolling_rate_{short}"].fillna(global_mean)

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **FEATURE CHECK: FARMER LAST ADOPTED**
Question it seeks to ask is, did this farmer adopt in their most recent prior training?

In [ ]:

last_training = prior_set.sort_values("training_day").groupby("farmer_name").last().reset_index()

temp_last = pd.DataFrame()
temp_last["farmer_last_adopted_07"] = train_set["farmer_name"].map(last_training.set_index("farmer_name")["adopted_within_07_days"])
temp_last["adopted_07"] = train_set["adopted_within_07_days"]

temp_last.groupby("farmer_last_adopted_07")["adopted_07"].agg(["mean", "size"])

,mean,size
farmer_last_adopted_07,,
0.0,0.015156,3167
1.0,0.230769,26


In [ ]:
last_training = prior_set.sort_values("training_day").groupby("farmer_name").last().reset_index()

for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    train_set[f"farmer_last_adopted_{short}"] = train_set["farmer_name"].map(last_training.set_index("farmer_name")[target]).fillna(0)

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

##  Farmer days since last adoption
 Seeks to answer: how many days since this farmer last adopted anything.

In [ ]:
last_adoption_date = prior_set[prior_set["adopted_within_07_days"] == 1].groupby("farmer_name")["training_day"].max()

temp_days = pd.DataFrame()
temp_days["days_since_last_adoption"] = (
    train_set["training_day"] - train_set["farmer_name"].map(last_adoption_date)
).dt.days
temp_days["adopted_07"] = train_set["adopted_within_07_days"]


temp_days["bucket"] = pd.cut(temp_days["days_since_last_adoption"], bins=[0, 30, 90, 180, 365, 9999])
temp_days.groupby("bucket")["adopted_07"].agg(["mean", "size"])

/tmp/ipykernel_399/3604089074.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp_days.groupby("bucket")["adopted_07"].agg(["mean", "size"])


,mean,size
bucket,,
"(0, 30]",0.000000,8
"(30, 90]",0.228571,35
"(90, 180]",0.000000,12
"(180, 365]",0.000000,27
"(365, 9999]",1.000000,2


# **FARMER+TOPIC**

In [ ]:
prior_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44882 entries, 0 to 44881
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       44882 non-null  object        
 1   farmer_name              44882 non-null  object        
 2   training_day             44882 non-null  datetime64[ns]
 3   gender                   44882 non-null  object        
 4   registration             44882 non-null  object        
 5   age                      44882 non-null  object        
 6   group_name               44882 non-null  object        
 7   belong_to_cooperative    44882 non-null  int64         
 8   county                   44882 non-null  object        
 9   subcounty                44882 non-null  object        
 10  ward                     44882 non-null  object        
 11  adopted_within_07_days   44882 non-null  int64         
 12  adopted_within_90_days   44882 n

## **SUBCOUNTY**

In [ ]:
prior_set.groupby("subcounty")["adopted_within_07_days"].agg(["mean","sum"])

,mean,sum
subcounty,,
SUB_bvliimx,0.000000,0
SUB_ccrxvdv,0.000000,0
SUB_ciorqql,0.000000,0
SUB_creyyqh,0.000000,0
SUB_ddnxwiw,0.000000,0
SUB_dhaaasn,0.013609,209
SUB_dtxnnxj,0.000000,0
SUB_forsajb,0.000000,0
SUB_hdsoyfn,0.008130,1


## **SUBCOUNTY FEATURE ENGINEERING**

In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    train_set[f"subcounty_adoption_{short}"] = smoothed_target_encoding(prior_set, train_set, "subcounty", target, m=20)

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 39 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **WARD**

In [ ]:
prior_set.groupby("ward")["adopted_within_07_days"].agg(["mean", "sum"]).sort_values("mean", ascending=False).head(20)

,mean,sum
ward,,
WRD_dhaaasnp,0.200000,4
WRD_dnforsaj,0.087282,70
WRD_ndkrworp,0.065217,3
WRD_usspxcre,0.058824,1
WRD_bbrifgee,0.042945,14
WRD_avjijsjd,0.036721,112
WRD_zfscqotw,0.036082,21
WRD_atkhhvon,0.031767,73
WRD_gwikueno,0.028388,150


## **WARD FEATURE ENGINEERING**

In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    train_set[f"ward_adoption_{short}"] = smoothed_target_encoding(prior_set, train_set, "ward", target, m=20)

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

In [ ]:
for short in ["07", "90", "120"]:
    global_mean = prior_set[f"adopted_within_{short}_days"].mean()
    train_set[f"ward_adoption_{short}"] = train_set[f"ward_adoption_{short}"].fillna(global_mean)

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **FEATURE ENGINEERING Topics list**

In [ ]:
# train_set.groupby(train_set["topics_list"].dt.month)["adopted_within_07_days"].mean()
train_set["topics_list"]

,topics_list
0,"[['Ndume App', 'Poultry Feeding']]"
1,"[['Poultry Housing'], ['Poultry Housing']]"
2,"[['Asili Fertilizer (Organic)', 'Biosecurity I..."
3,"[['Poultry Products'], ['Record Keeping In Dai..."
4,"[['Ndume App', 'Poultry Feeding']]"
...,...
13531,[['Digital Finance With Kcb']]
13532,[['Digital Finance With Kcb']]
13533,[['Digital Finance With Kcb']]
13534,[['Digital Finance With Kcb']]


In [ ]:
import ast
train_set["topics_list"].apply(lambda x: len(ast.literal_eval(x))).value_counts()

,count
topics_list,
1,6712
3,2283
2,2242
4,1836
5,325
6,58
8,26
7,20
12,13


In [ ]:
data_dict

,column_name,description
0,ID,unique identifier for each farmer entry
1,gender,Gender of the farmer
2,age,Age category of the farmer
3,registration,Registration method
4,belong_to_cooperative,Whether the farmer belongs to a cooperative (1...
5,county,County of residence
6,subcounty,Sub-county of residence
7,ward,Ward of residence
8,trainer,List of Trainers who delivered the training on...
9,topics,List of List of possible training topics


In [ ]:
prior_set["topics_list"].apply(lambda x: len(ast.literal_eval(x))).value_counts()

,count
topics_list,
1,43329
2,538
3,284
4,100
6,74
5,65
9,55
7,50
8,46


In [ ]:
temp = pd.DataFrame()
temp["num_sessions"] = train_set["topics_list"].apply(lambda x: len(ast.literal_eval(x)))
temp["adopted_07"] = train_set["adopted_within_07_days"]
temp.groupby("num_sessions")["adopted_07"].agg(["mean", "size"]).head(20)

,mean,size
num_sessions,,
1,0.010131,6712
2,0.011151,2242
3,0.011389,2283
4,0.007081,1836
5,0.006154,325
6,0.137931,58
7,0.250000,20
8,0.038462,26
9,0.250000,12


In [ ]:
temp2 = pd.DataFrame()
temp2["num_unique_topics"] = train_set["topics_list"].apply(lambda x: len(set([t for s in ast.literal_eval(x) for t in s])))
temp2["adopted_07"] = train_set["adopted_within_07_days"]
temp2.groupby("num_unique_topics")["adopted_07"].agg(["mean", "size"]).head(20)

,mean,size
num_unique_topics,,
1,0.008883,3152
2,0.018281,3118
3,0.011775,3397
4,0.002157,2318
5,0.002445,409
6,0.002604,384
7,0.008696,115
8,0.006135,163
9,0.023364,214


**Num topics and num sessions are not influential for adoption**

In [ ]:
prior_exploded = prior_set.copy()
prior_exploded["topics_list"] = prior_exploded["topics_list"].apply(lambda x: ast.literal_eval(x))
prior_exploded["topics_list"] = prior_exploded["topics_list"].apply(lambda x: [t for s in x for t in s] if isinstance(x[0], list) else x)
prior_exploded = prior_exploded.explode("topics_list")
prior_exploded.groupby("topics_list")["adopted_within_07_days"].agg(["mean", "size"]).sort_values("mean", ascending=False).tail(20)

,mean,size
topics_list,,
Selling To Choice Meats,0.0,116
Prevent And Treat Cows Against Worms,0.0,4
Production Of Quality Milk,0.0,139
Poultry Mngt Practices,0.0,1088
Sheep & Goats Management,0.0,67
Sheep & Goat Rearing.,0.0,107
Sheep & Goat Rearing,0.0,126
Sheep And Goat Management Practices,0.0,581
Some Reasons Why Ai Fails And Solutions,0.0,30


## **FEATURE ENGINEERING**

In [ ]:
topic_lookup_07 = prior_exploded.groupby("topics_list")["adopted_within_07_days"].mean()
def get_topic_rate(topics_str, lookup):
    parsed = ast.literal_eval(topics_str)
    flat = [t for s in parsed for t in s] if isinstance(parsed[0], list) else parsed
    rates = [lookup.get(t, lookup.mean()) for t in flat]
    return sum(rates) / len(rates)
train_set["topic_adoption_rate_07"] = train_set["topics_list"].apply(lambda x: get_topic_rate(x, topic_lookup_07))

topic_lookup_90 = prior_exploded.groupby("topics_list")["adopted_within_90_days"].mean()
train_set["topic_adoption_rate_90"] = train_set["topics_list"].apply(lambda x: get_topic_rate(x, topic_lookup_90))


topic_lookup_120 = prior_exploded.groupby("topics_list")["adopted_within_120_days"].mean()
train_set["topic_adoption_rate_120"] = train_set["topics_list"].apply(lambda x: get_topic_rate(x, topic_lookup_120))

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 45 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

FEATURE

In [ ]:
print("prior_exploded" in dir())
prior_exploded.shape

True


(53746, 17)

In [ ]:
farmer_topic = prior_exploded.groupby(["farmer_name", "topics_list"])["adopted_within_07_days"].max().reset_index()
farmer_topic.columns = ["farmer_name", "topic", "ever_adopted"]


def farmer_topic_adopted(row, lookup):
    try:
        parsed = ast.literal_eval(row["topics_list"])
        flat = [t for s in parsed for t in s] if isinstance(parsed[0], list) else parsed
        results = [lookup.get((row["farmer_name"], t), 0) for t in flat]
        return max(results)
    except:
        return 0

farmer_topic_lookup = farmer_topic.set_index(["farmer_name", "topic"])["ever_adopted"].to_dict()

temp_ft = pd.DataFrame()
temp_ft["farmer_topic_ever_adopted"] = train_set.apply(
    lambda row: farmer_topic_adopted(row, farmer_topic_lookup), axis=1
)
temp_ft["adopted_07"] = train_set["adopted_within_07_days"]

temp_ft.groupby("farmer_topic_ever_adopted")["adopted_07"].agg(["mean", "size"])

,mean,size
farmer_topic_ever_adopted,,
0,0.011089,13527
1,0.333333,9


In [ ]:
prior_exploded.groupby("topics_list")["adopted_within_07_days"].agg(["mean", "size"]).sort_values("mean", ascending=False).head(15)

,mean,size
topics_list,,
Poultry Health With Biodeal,0.149798,247
Importance Of Mineral Supplementation,0.108731,1214
Benefits Of Sistema Biogas,0.090909,22
Importance Of Vaccinating Against East Coast Fever (Ecf),0.090909,88
Dairy Nutrition With Tyari,0.087459,606
Harvesting & Post-Harvest Handling.,0.086957,23
Herd. Health. Management,0.086449,428
Poultry Health Management. Practices,0.076125,1156
Poultry Feeding With Tyari,0.075682,806


In [ ]:
high_topics = [
    "Poultry Health With Biodeal",
    "Importance Of Mineral Supplementation",
    "Dairy Nutrition With Tyari",
    "Herd. Health. Management",
    "Poultry Health Management. Practices",
    "Poultry Feeding With Tyari",
    "Diseases In Dairy Farming",
    "Poultry Health Management",
    "Dairy Health Management"
]

temp_high = pd.DataFrame()
temp_high["is_high_adoption_topic"] = train_set["topics_list"].apply(
    lambda x: int(any(t in high_topics for s in ast.literal_eval(x) for t in s))
)
temp_high["adopted_07"] = train_set["adopted_within_07_days"]

temp_high.groupby("is_high_adoption_topic")["adopted_07"].agg(["mean", "size"])

,mean,size
is_high_adoption_topic,,
0,0.004597,11748
1,0.055369,1788


In [ ]:
train_set["is_high_adoption_topic"] = train_set["topics_list"].apply(lambda x: int(any(t in high_topics for s in ast.literal_eval(x) for t in s)))

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 46 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **TIME FEATURES**

In [ ]:
train_set.groupby(train_set["training_day"].dt.month)["adopted_within_07_days"].agg(["mean","size"])
# train_set.groupby(train_set["training_day"].dt.dayofweek)["adopted_within_07_days"].mean()

,mean,size
training_day,,
1,0.009735,1130
2,0.010156,1477
3,0.019048,1155
4,0.009434,1484
5,0.004963,806
6,0.008183,1222
7,0.001088,919
8,0.010101,1188
9,0.019833,958


In [ ]:
train_set["training_month"] = train_set["training_day"].dt.month
train_set["training_dayofweek"] = train_set["training_day"].dt.dayofweek
train_set["is_long_rains"] = train_set["training_month"].isin([3,4,5]).astype(int)
train_set["is_short_rains"] = train_set["training_month"].isin([10,11,12]).astype(int)

In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 50 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

## **RECENCY**

In [ ]:
prior_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44882 entries, 0 to 44881
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       44882 non-null  object        
 1   farmer_name              44882 non-null  object        
 2   training_day             44882 non-null  datetime64[ns]
 3   gender                   44882 non-null  object        
 4   registration             44882 non-null  object        
 5   age                      44882 non-null  object        
 6   group_name               44882 non-null  object        
 7   belong_to_cooperative    44882 non-null  int64         
 8   county                   44882 non-null  object        
 9   subcounty                44882 non-null  object        
 10  ward                     44882 non-null  object        
 11  adopted_within_07_days   44882 non-null  int64         
 12  adopted_within_90_days   44882 n

In [ ]:

last_training = prior_set.groupby("farmer_name")["training_day"].max()

temp3 = pd.DataFrame()
temp3["last_prior_training"] = train_set["farmer_name"].map(last_training)
temp3["training_day"] = train_set["training_day"]
temp3["days_since"] = (temp3["training_day"] - temp3["last_prior_training"]).dt.days
temp3["adopted_07"] = train_set["adopted_within_07_days"]

temp3["recency_bucket"] = pd.cut(temp3["days_since"], bins=[0,30,90,180,365,9999])
temp3.groupby("recency_bucket")["adopted_07"].agg(["mean","size"])

/tmp/ipykernel_399/2078714596.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp3.groupby("recency_bucket")["adopted_07"].agg(["mean","size"])


,mean,size
recency_bucket,,
"(0, 30]",0.012027,582
"(30, 90]",0.018086,1327
"(90, 180]",0.012539,638
"(180, 365]",0.024648,568
"(365, 9999]",0.012821,78


# OTHER FEATURES

In [ ]:
# train_set.groupby("registration")["adopted_within_07_days"].mean()
train_set.groupby("age")["adopted_within_07_days"].mean().sort_values(ascending=False).head(10)

,adopted_within_07_days
age,
Below 35,0.014402
Above 35,0.010528


In [ ]:
train_set["is_ussd"] = (train_set["registration"].str.lower() == "ussd").astype(int)
train_set["is_female"] = (train_set["gender"].str.lower() == "female").astype(int)
train_set["is_above_35"] = (train_set["age"].str.lower() == "above 35").astype(int)

In [ ]:
train_set.groupby("belong_to_cooperative")["adopted_within_07_days"].mean()

,adopted_within_07_days
belong_to_cooperative,
0,0.009218
1,0.023859


In [ ]:
prior_set.groupby(["belong_to_cooperative", "county"])["adopted_within_07_days"].agg(["mean", "size"])

mean   size
belong_to_cooperative county                     
0                     CNT_fhdsoy  0.035881   3428
                      CNT_hfhimd  0.000000    130
                      CNT_lpotuu  0.000000    205
                      CNT_mggvna  0.013274    226
                      CNT_mvqqmw  0.028388   5284
                      CNT_rihpna  0.000000   2777
                      CNT_vvuukv  0.010638  19646
                      CNT_yljqnl  0.000792   8834
1                     CNT_fhdsoy  0.041586   4112
                      CNT_fnywiv  0.000000      1
                      CNT_lpotuu  0.500000      2
                      CNT_mggvna  0.000000    140
                      CNT_rihpna  0.000000     87
                      CNT_vvuukv  0.000000     10

In [ ]:
group_topic_rates = prior_exploded.groupby(["group_name", "topics_list"])["adopted_within_07_days"].agg(["mean", "size"]).reset_index()
group_topic_rates.columns = ["group_name", "topic", "adoption_rate", "count"]

group_topic_rates.sort_values("adoption_rate", ascending=False).head(10)

,group_name,topic,adoption_rate,count
4938,GRP_wftrnca,Dairy Nutrition With Tyari,1.0,2
4939,GRP_wftrnca,Importance Of Vaccinations And Record,1.0,1
4940,GRP_wftrnca,Livestock Management Practices,1.0,1
4941,GRP_wftrnca,Poultry Feeding With Tyari,1.0,2
5371,GRP_yiavgqw,Diseases In Dairy Farming,1.0,1
5370,GRP_yiavgqw,Dairy Health Management,1.0,1
1674,GRP_iegtpjx,Livestock Management Practices.,1.0,1
1671,GRP_iegtpjx,How To Manage Pest And Diseases At Harvesting ...,1.0,1
1669,GRP_iegtpjx,Control Of External Parasites With Dominex.,1.0,1
1668,GRP_iegtpjx,Benfits Of Sistema Biogas,1.0,1


In [ ]:
print(group_topic_rates["count"].describe())
print(group_topic_rates[group_topic_rates["count"] >= 5].shape[0], "group-topic pairs with 5+ observations")
print(group_topic_rates[group_topic_rates["count"] >= 10].shape[0], "group-topic pairs with 10+ observations")

count    5759.000000
mean        9.332523
std        13.915453
min         1.000000
25%         1.000000
50%         4.000000
75%        11.000000
max       148.000000
Name: count, dtype: float64
2597 group-topic pairs with 5+ observations
1625 group-topic pairs with 10+ observations


In [ ]:

group_topic_lookup = group_topic_rates.set_index(["group_name", "topic"])["adoption_rate"].to_dict()

def get_group_topic_rate(row, lookup):
    try:
        parsed = ast.literal_eval(row["topics_list"])
        flat = [t for s in parsed for t in s] if isinstance(parsed[0], list) else parsed
        rates = [lookup.get((row["group_name"], t), None) for t in flat]
        rates = [r for r in rates if r is not None]
        if len(rates) == 0:
            return None
        return sum(rates) / len(rates)
    except:
        return None

temp_gt = pd.DataFrame()
temp_gt["group_topic_rate"] = train_set.apply(
    lambda row: get_group_topic_rate(row, group_topic_lookup), axis=1
)
temp_gt["adopted_07"] = train_set["adopted_within_07_days"]

print(f"Non-null: {temp_gt['group_topic_rate'].notna().sum()}")
temp_gt.groupby(pd.cut(temp_gt["group_topic_rate"], bins=5))["adopted_07"].agg(["mean", "size"])

Non-null: 12464


/tmp/ipykernel_399/994998847.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp_gt.groupby(pd.cut(temp_gt["group_topic_rate"], bins=5))["adopted_07"].agg(["mean", "size"])


,mean,size
group_topic_rate,,
"(-0.001, 0.2]",0.007589,12255
"(0.2, 0.4]",0.236486,148
"(0.4, 0.6]",0.226415,53
"(0.6, 0.8]",0.000000,1
"(0.8, 1.0]",0.285714,7


In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    rates = prior_exploded.groupby(["group_name", "topics_list"])[target].mean()
    lookup = rates.to_dict()
    global_mean = prior_set[target].mean()

    train_set[f"group_topic_rate_{short}"] = train_set.apply(
        lambda row, l=lookup: get_group_topic_rate(row, l), axis=1
    ).fillna(global_mean)

print(train_set[["group_topic_rate_07", "group_topic_rate_90", "group_topic_rate_120"]].describe())

       group_topic_rate_07  group_topic_rate_90  group_topic_rate_120
count         13536.000000         13536.000000          13536.000000
mean              0.008990             0.019292              0.029174
std               0.050852             0.084083              0.106354
min               0.000000             0.000000              0.000000
25%               0.000000             0.000000              0.000000
50%               0.000000             0.000000              0.000000
75%               0.000000             0.000000              0.000000
max               1.000000             1.000000              1.000000


FEATURE CHECK:TRAINER TOPIC ADOPTION RATE

In [ ]:
temp_tt = pd.DataFrame()

trainer_topic_rates = prior_exploded.groupby(["trainer", "topics_list"])["adopted_within_07_days"].agg(["mean", "size"]).reset_index()
trainer_topic_rates.columns = ["trainer", "topic", "adoption_rate", "count"]

print(trainer_topic_rates["count"].describe())
print(trainer_topic_rates[trainer_topic_rates["count"] >= 5].shape[0], "trainer-topic pairs with 5+ observations")

trainer_topic_lookup = trainer_topic_rates.set_index(["trainer", "topic"])["adoption_rate"].to_dict()

def get_trainer_topic_rate(row, lookup):
    try:
        parsed = ast.literal_eval(row["topics_list"])
        flat = [t for s in parsed for t in s] if isinstance(parsed[0], list) else parsed
        rates = [lookup.get((row["trainer"], t), None) for t in flat]
        rates = [r for r in rates if r is not None]
        if len(rates) == 0:
            return None
        return sum(rates) / len(rates)
    except:
        return None

temp_tt["trainer_topic_rate"] = train_set.apply(
    lambda row: get_trainer_topic_rate(row, trainer_topic_lookup), axis=1
)
temp_tt["adopted_07"] = train_set["adopted_within_07_days"]

print(f"Non-null: {temp_tt['trainer_topic_rate'].notna().sum()}")
temp_tt.groupby(pd.cut(temp_tt["trainer_topic_rate"], bins=5))["adopted_07"].agg(["mean", "size"])

count     353.000000
mean      152.254958
std       191.277902
min         1.000000
25%        22.000000
50%        87.000000
75%       199.000000
max      1008.000000
Name: count, dtype: float64
329 trainer-topic pairs with 5+ observations
Non-null: 13518


/tmp/ipykernel_399/3692070466.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp_tt.groupby(pd.cut(temp_tt["trainer_topic_rate"], bins=5))["adopted_07"].agg(["mean", "size"])


,mean,size
trainer_topic_rate,,
"(-0.000253, 0.0505]",0.004506,12650
"(0.0505, 0.101]",0.081232,714
"(0.101, 0.152]",0.238532,109
"(0.152, 0.202]",0.243902,41
"(0.202, 0.253]",0.500000,4


In [ ]:
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    rates = prior_exploded.groupby(["trainer", "topics_list"])[target].mean()
    lookup = rates.to_dict()
    global_mean = prior_set[target].mean()

    train_set[f"trainer_topic_rate_{short}"] = train_set.apply(lambda row, l=lookup: get_trainer_topic_rate(row, l), axis=1).fillna(global_mean)

print(train_set[["trainer_topic_rate_07", "trainer_topic_rate_90", "trainer_topic_rate_120"]].describe())

       trainer_topic_rate_07  trainer_topic_rate_90  trainer_topic_rate_120
count           13536.000000           13536.000000            13536.000000
mean                0.011338               0.021011                0.037769
std                 0.025272               0.043895                0.052832
min                 0.000000               0.000000                0.000000
25%                 0.000000               0.000000                0.000000
50%                 0.000000               0.000000                0.010614
75%                 0.009853               0.026768                0.062521
max                 0.252525               0.848485                0.848485


In [ ]:
group_size = prior_set.groupby("group_name")["farmer_name"].nunique().reset_index()
group_size.columns = ["group_name", "group_size"]

temp_gs = pd.DataFrame()
temp_gs["group_size"] = train_set["group_name"].map(group_size.set_index("group_name")["group_size"])
temp_gs["adopted_07"] = train_set["adopted_within_07_days"]

print(f"Non-null: {temp_gs['group_size'].notna().sum()}")
temp_gs.groupby(pd.cut(temp_gs["group_size"], bins=[0,10,50,100,500,9999]))["adopted_07"].agg(["mean","size"])

Non-null: 12616


/tmp/ipykernel_399/3805809373.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp_gs.groupby(pd.cut(temp_gs["group_size"], bins=[0,10,50,100,500,9999]))["adopted_07"].agg(["mean","size"])


,mean,size
group_size,,
"(0, 10]",0.013417,5739
"(10, 50]",0.010582,4158
"(50, 100]",0.010356,1545
"(100, 500]",0.004259,1174
"(500, 9999]",NaN,0


Chcecked from feat_imp after model building and saw how highly has_topic_trained_on ranked relative to others. So we are exploiting the feature

In [ ]:
# does being in a high adoption group matter more when trained on topic?
temp_int = pd.DataFrame()
temp_int["feature"] = train_set["has_topic_trained_on"] * train_set["group_adoption_07"]
temp_int["adopted_07"] = train_set["adopted_within_07_days"]
temp_int.groupby(pd.cut(temp_int["feature"], bins=5))["adopted_07"].agg(["mean","size"])

/tmp/ipykernel_399/1655471628.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp_int.groupby(pd.cut(temp_int["feature"], bins=5))["adopted_07"].agg(["mean","size"])


,mean,size
feature,,
"(-0.000432, 0.0864]",0.00989,13347
"(0.0864, 0.173]",0.10989,182
"(0.173, 0.259]",NaN,0
"(0.259, 0.346]",0.00000,6
"(0.346, 0.432]",1.00000,1


In [ ]:

# train_set["topic_group_interaction"] = train_set["has_topic_trained_on"] * train_set["group_adoption_07"]


In [ ]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 59 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       13536 non-null  object        
 1   farmer_name              13536 non-null  object        
 2   training_day             13536 non-null  datetime64[ns]
 3   gender                   13536 non-null  object        
 4   registration             13536 non-null  object        
 5   age                      13536 non-null  object        
 6   group_name               13536 non-null  object        
 7   belong_to_cooperative    13536 non-null  int64         
 8   county                   13536 non-null  object        
 9   subcounty                13536 non-null  object        
 10  ward                     13536 non-null  object        
 11  adopted_within_07_days   13536 non-null  int64         
 12  adopted_within_90_days   13536 n

### **TEST SET**

In [ ]:
test_set["trainer"].value_counts().head(5)
# test_set["topics_list"].iloc[0]

,count
trainer,
['TRA_suiifsur'],1976
['TRA_ubcgvofe'],1151
['TRA_gertumxc'],1078
['TRA_kkzpfdtu'],734
['TRA_rkvyofbh'],660


In [ ]:
len(history_set)

58418

In [ ]:
test_set["trainer"] = test_set["trainer"].str.strip("[]'")
test_set["training_day"] = pd.to_datetime(test_set["training_day"])

In [ ]:
test_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5621 entries, 0 to 5620
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   ID                     5621 non-null   object        
 1   farmer_name            5621 non-null   object        
 2   training_day           5621 non-null   datetime64[ns]
 3   gender                 5621 non-null   object        
 4   registration           5621 non-null   object        
 5   age                    5621 non-null   object        
 6   group_name             5621 non-null   object        
 7   belong_to_cooperative  5621 non-null   int64         
 8   county                 5621 non-null   object        
 9   subcounty              5621 non-null   object        
 10  ward                   5621 non-null   object        
 11  has_topic_trained_on   5621 non-null   int64         
 12  trainer                5621 non-null   object        
 13  top

In [ ]:
# county
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    test_set[f"county_adoption_{short}"] = test_set["county"].map(history_set.groupby("county")[target].mean())

# trainer
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    test_set[f"trainer_adoption_{short}"] = smoothed_target_encoding(history_set, test_set, "trainer", target, m=20)

# group
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    test_set[f"group_adoption_{short}"] = smoothed_target_encoding(history_set, test_set, "group_name", target, m=20)

# subcounty
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    test_set[f"subcounty_adoption_{short}"] = smoothed_target_encoding(history_set, test_set, "subcounty", target, m=20)

# ward
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    test_set[f"ward_adoption_{short}"] = smoothed_target_encoding(history_set, test_set, "ward", target, m=20)

# fix nulls from smoothed encoding
for short in ["07", "90", "120"]:
    global_mean = history_set[f"adopted_within_{short}_days"].mean()
    test_set[f"group_adoption_{short}"] = test_set[f"group_adoption_{short}"].fillna(global_mean)
    test_set[f"trainer_adoption_{short}"] = test_set[f"trainer_adoption_{short}"].fillna(global_mean)
    test_set[f"ward_adoption_{short}"] = test_set[f"ward_adoption_{short}"].fillna(global_mean)
    test_set[f"subcounty_adoption_{short}"] = test_set[f"subcounty_adoption_{short}"].fillna(global_mean)

# farmer features with date filtering
all_history = pd.concat([prior_set, train_set], ignore_index=True)

temp = test_set[["farmer_name", "training_day"]].copy().reset_index()
merged = temp.merge(
    all_history[["farmer_name", "training_day", "adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]],
    on="farmer_name", suffixes=("_main", "_hist")
)
merged = merged[merged["training_day_hist"] < merged["training_day_main"]]

agg = merged.groupby("index").agg(
    farmer_training_count=("adopted_within_07_days", "count"),
    farmer_ever_adopted_07=("adopted_within_07_days", "sum"),
    farmer_ever_adopted_90=("adopted_within_90_days", "sum"),
    farmer_ever_adopted_120=("adopted_within_120_days", "sum")
)
agg["farmer_ever_adopted_07"] = (agg["farmer_ever_adopted_07"] > 0).astype(int)
agg["farmer_ever_adopted_90"] = (agg["farmer_ever_adopted_90"] > 0).astype(int)
agg["farmer_ever_adopted_120"] = (agg["farmer_ever_adopted_120"] > 0).astype(int)

test_set = test_set.join(agg)
for col in ["farmer_training_count", "farmer_ever_adopted_07", "farmer_ever_adopted_90", "farmer_ever_adopted_120"]:
    test_set[col] = test_set[col].fillna(0)

# farmer rolling rate
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    test_set[f"farmer_rolling_rate_{short}"] = smoothed_target_encoding(
        all_history, test_set, "farmer_name", target, m=20
    )

# farmer last adopted
last_training_hist = all_history.sort_values("training_day").groupby("farmer_name").last().reset_index()
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    test_set[f"farmer_last_adopted_{short}"] = test_set["farmer_name"].map(
        last_training_hist.set_index("farmer_name")[target]
    ).fillna(0)

# topic features
history_exploded = all_history.copy()
history_exploded["topics_list"] = history_exploded["topics_list"].apply(lambda x: ast.literal_eval(x))
history_exploded["topics_list"] = history_exploded["topics_list"].apply(lambda x: [t for s in x for t in s] if isinstance(x[0], list) else x)
history_exploded = history_exploded.explode("topics_list")

for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    topic_lookup = history_exploded.groupby("topics_list")[target].mean()
    test_set[f"topic_adoption_rate_{short}"] = test_set["topics_list"].apply(lambda x: get_topic_rate(x, topic_lookup))

# is_high_adoption_topic
high_topics = [
    "Poultry Health With Biodeal",
    "Importance Of Mineral Supplementation",
    "Dairy Nutrition With Tyari",
    "Herd. Health. Management",
    "Poultry Health Management. Practices",
    "Poultry Feeding With Tyari",
    "Diseases In Dairy Farming",
    "Poultry Health Management",
    "Dairy Health Management"
]
test_set["is_high_adoption_topic"] = test_set["topics_list"].apply(
    lambda x: int(any(t in high_topics for s in ast.literal_eval(x) for t in s))
)

# group_topic_rate
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    rates = history_exploded.groupby(["group_name", "topics_list"])[target].mean()
    lookup = rates.to_dict()
    global_mean = all_history[target].mean()
    test_set[f"group_topic_rate_{short}"] = test_set.apply(
        lambda row, l=lookup: get_group_topic_rate(row, l), axis=1
    ).fillna(global_mean)

# trainer_topic_rate
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    rates = history_exploded.groupby(["trainer", "topics_list"])[target].mean()
    lookup = rates.to_dict()
    global_mean = all_history[target].mean()
    test_set[f"trainer_topic_rate_{short}"] = test_set.apply(
        lambda row, l=lookup: get_trainer_topic_rate(row, l), axis=1
    ).fillna(global_mean)

# time features
test_set["training_month"] = test_set["training_day"].dt.month
test_set["training_dayofweek"] = test_set["training_day"].dt.dayofweek
test_set["is_long_rains"] = test_set["training_month"].isin([3,4,5]).astype(int)
test_set["is_short_rains"] = test_set["training_month"].isin([10,11,12]).astype(int)

# simple features
test_set["is_ussd"] = (test_set["registration"].str.lower() == "ussd").astype(int)
test_set["is_female"] = (test_set["gender"].str.lower() == "female").astype(int)
test_set["is_above_35"] = (test_set["age"].str.lower() == "above 35").astype(int)

# test_set["topic_group_interaction"] = test_set["has_topic_trained_on"] * test_set["group_adoption_07"]


print(test_set.shape)
print(test_set.isnull().sum().sum())

(5621, 56)
6285


In [ ]:
for short in ["07", "90", "120"]:
    global_mean = all_history[f"adopted_within_{short}_days"].mean()
    test_set[f"farmer_rolling_rate_{short}"] = test_set[f"farmer_rolling_rate_{short}"].fillna(global_mean)

test_set.isnull().sum().sum()

np.int64(0)

In [ ]:
cols_to_drop = ["ID", "farmer_name", "training_day", "gender", "registration", "age",
                "group_name", "county", "subcounty", "ward", "trainer", "topics_list"]
test_ids = test_set["ID"].copy()
train_features = train_set.drop(columns=cols_to_drop)
test_features = test_set.drop(columns=cols_to_drop)
target_cols = ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]
X = train_features.drop(columns=target_cols)
y = train_features[target_cols]

X.shape, test_features.shape

((13536, 44), (5621, 44))

In [ ]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, roc_auc_score
import xgboost as xgb


date_order = train_set["training_day"].argsort().values
X_sorted = X.iloc[date_order].reset_index(drop=True)
y_sorted = y.iloc[date_order].reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=4)
test_features = test_features[X_sorted.columns]


def calculate_weighted_score(target_07_auc, target_07_logloss, target_90_auc, target_90_logloss, target_120_auc, target_120_logloss):
    SCALING = 0.56926
    target_07_logloss_norm = 1 - (target_07_logloss / SCALING)
    target_90_logloss_norm = 1 - (target_90_logloss / SCALING)
    target_120_logloss_norm = 1 - (target_120_logloss / SCALING)
    return (target_07_auc * 0.15 + target_07_logloss_norm * 0.65 + target_90_auc * 0.05 + target_90_logloss_norm * 0.05 + target_120_auc * 0.05 + target_120_logloss_norm * 0.05)

In [ ]:
models_cv_cat = {}
cv_scores_cat = {}

for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    fold_ll, fold_auc, test_preds = [], [], []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sorted)):
        model = CatBoostClassifier(
            iterations=500, learning_rate=0.05, depth=6,
            scale_pos_weight=45, random_seed=42, verbose=0
        )
        model.fit(X_sorted.iloc[train_idx], y_sorted[target].iloc[train_idx])
        val_preds = model.predict_proba(X_sorted.iloc[val_idx])[:,1]
        fold_ll.append(log_loss(y_sorted[target].iloc[val_idx], val_preds))
        fold_auc.append(roc_auc_score(y_sorted[target].iloc[val_idx], val_preds))
        test_preds.append(model.predict_proba(test_features)[:,1])
        print(f"{short} Fold {fold+1}: LogLoss={fold_ll[-1]:.4f}, AUC={fold_auc[-1]:.4f}")

    cv_scores_cat[f"{short}_ll"] = np.mean(fold_ll)
    cv_scores_cat[f"{short}_auc"] = np.mean(fold_auc)
    models_cv_cat[target] = np.mean(test_preds, axis=0)
    print(f"\n{short} CAT CV: LogLoss={cv_scores_cat[f'{short}_ll']:.4f}, AUC={cv_scores_cat[f'{short}_auc']:.4f}\n")

cat_weighted = calculate_weighted_score(
    cv_scores_cat["07_auc"], cv_scores_cat["07_ll"],
    cv_scores_cat["90_auc"], cv_scores_cat["90_ll"],
    cv_scores_cat["120_auc"], cv_scores_cat["120_ll"]
)
print(f"CatBoost CV weighted score: {cat_weighted:.4f}")

07 Fold 1: LogLoss=0.0581, AUC=0.9752
07 Fold 2: LogLoss=0.0325, AUC=0.9773
07 Fold 3: LogLoss=0.0575, AUC=0.9622
07 Fold 4: LogLoss=0.0831, AUC=0.9398

07 CAT CV: LogLoss=0.0578, AUC=0.9636

90 Fold 1: LogLoss=0.0749, AUC=0.9300
90 Fold 2: LogLoss=0.0268, AUC=0.9752
90 Fold 3: LogLoss=0.0638, AUC=0.9366
90 Fold 4: LogLoss=0.0814, AUC=0.9315

90 CAT CV: LogLoss=0.0617, AUC=0.9433

120 Fold 1: LogLoss=0.0971, AUC=0.9462
120 Fold 2: LogLoss=0.0544, AUC=0.9666
120 Fold 3: LogLoss=0.0686, AUC=0.9548
120 Fold 4: LogLoss=0.1059, AUC=0.8931

120 CAT CV: LogLoss=0.0815, AUC=0.9402

CatBoost CV weighted score: 0.9102


In [ ]:
models_cv_xgb = {}
cv_scores_xgb = {}

for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    short = target.replace("adopted_within_", "").replace("_days", "")
    fold_ll, fold_auc, test_preds = [], [], []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sorted)):
        model = xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            scale_pos_weight=45, random_state=42,
            eval_metric="logloss", verbosity=0
        )
        model.fit(X_sorted.iloc[train_idx], y_sorted[target].iloc[train_idx])
        val_preds = model.predict_proba(X_sorted.iloc[val_idx])[:,1]
        fold_ll.append(log_loss(y_sorted[target].iloc[val_idx], val_preds))
        fold_auc.append(roc_auc_score(y_sorted[target].iloc[val_idx], val_preds))
        test_preds.append(model.predict_proba(test_features)[:,1])
        print(f"{short} Fold {fold+1}: LogLoss={fold_ll[-1]:.4f}, AUC={fold_auc[-1]:.4f}")

    cv_scores_xgb[f"{short}_ll"] = np.mean(fold_ll)
    cv_scores_xgb[f"{short}_auc"] = np.mean(fold_auc)
    models_cv_xgb[target] = np.mean(test_preds, axis=0)
    print(f"\n{short} XGB CV: LogLoss={cv_scores_xgb[f'{short}_ll']:.4f}, AUC={cv_scores_xgb[f'{short}_auc']:.4f}\n")

xgb_weighted = calculate_weighted_score(
    cv_scores_xgb["07_auc"], cv_scores_xgb["07_ll"],
    cv_scores_xgb["90_auc"], cv_scores_xgb["90_ll"],
    cv_scores_xgb["120_auc"], cv_scores_xgb["120_ll"]
)
print(f"XGBoost CV weighted score: {xgb_weighted:.4f}")

07 Fold 1: LogLoss=0.0803, AUC=0.9643
07 Fold 2: LogLoss=0.0451, AUC=0.9792
07 Fold 3: LogLoss=0.0572, AUC=0.9688
07 Fold 4: LogLoss=0.1100, AUC=0.9164

07 XGB CV: LogLoss=0.0731, AUC=0.9572

90 Fold 1: LogLoss=0.0868, AUC=0.8846
90 Fold 2: LogLoss=0.0320, AUC=0.9588
90 Fold 3: LogLoss=0.0774, AUC=0.9634
90 Fold 4: LogLoss=0.1149, AUC=0.9180

90 XGB CV: LogLoss=0.0778, AUC=0.9312

120 Fold 1: LogLoss=0.1497, AUC=0.9388
120 Fold 2: LogLoss=0.0499, AUC=0.9694
120 Fold 3: LogLoss=0.0646, AUC=0.9413
120 Fold 4: LogLoss=0.1080, AUC=0.9155

120 XGB CV: LogLoss=0.0931, AUC=0.9412

XGBoost CV weighted score: 0.8887


In [ ]:
def make_submission(preds_dict, filename):
    sub = pd.read_csv('/content/SampleSubmission.csv')
    sub["ID"] = test_ids.values
    sub['Target_07_AUC']      = preds_dict["adopted_within_07_days"]
    sub['Target_90_AUC']      = preds_dict["adopted_within_90_days"]
    sub['Target_120_AUC']     = preds_dict["adopted_within_120_days"]
    sub['Target_07_LogLoss']  = sub['Target_07_AUC']
    sub['Target_90_LogLoss']  = sub['Target_90_AUC']
    sub['Target_120_LogLoss'] = sub['Target_120_AUC']
    sub.to_csv(filename, index=False)

In [ ]:
ensemble_best = {}
for target in ["adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days"]:
    ensemble_best[target] = 0.90 * models_cv_cat[target] + 0.10 * models_cv_xgb[target]


make_submission(ensemble_best, "submission_cat90_xgb10_s45.csv")